In [5]:
import os
import json
import warnings
import numpy as np
import pandas as pd

from data.db import db

warnings.filterwarnings('ignore')

# ════════════════════════════════════════════════
# ✏️  여기만 수정하세요
VERSION          = 'v2'          # 'v1' / 'v2' / 'v3'
CUTOFF_DATE      = '2025-11-01'  # 이 날짜 이전 개봉작만 사용
STAR_POWER_TOP_N = 3             # 주연 배우 Star Power 상위 N명

HIT_BINS = [
    0,
    3_000_000,
    float('inf')
]

HIT_LABELS = [0, 1] # 일반 : 0, 흥행 : 1
# ════════════════════════════════════════════════

# ── v1 컬럼 정의 ──
ID_COLS     = ['movie_id', 'title']
TARGET_COLS = ['total_audience', 'log_audience', 'hit_class']

V1_FEATURE_COLS = (
    ['runtime', 'rating_encoded', 'is_korean', 'genre']            # 메타
  + ['open_date','open_month', 'open_day_of_week', 'is_summer', 'is_winter',
     'is_holiday_release', 'holiday_nearby_count']                 # 시간/계절
  + ['director_avg_audi', 'director_movie_count',
     'lead_actor_avg_audi', 'lead_actor_movie_count',
     'cast_max_star_power']                                        # Star Power
  + ['distributor_avg_audi', 'distributor_movie_count',
     'producer_avg_audi', 'producer_movie_count']                  # Brand Power
  + ['same_week_releases', 'market_avg_audi_7d']                   # 경쟁 환경
)

OUTPUT_DIR  = 'data/processed'
OUTPUT_PATH = f'{OUTPUT_DIR}/feature_table_{VERSION}.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# hit_class 설정을 JSON으로 저장 (ML 노트북에서 참조용)
with open('hit_config.json', 'w', encoding='utf-8') as f:
    json.dump({'HIT_BINS': HIT_BINS, 'HIT_LABELS': HIT_LABELS}, f, ensure_ascii=False)


In [6]:
V1_PATH = f'{OUTPUT_DIR}/feature_table_v1.csv'

if VERSION != 'v1' and os.path.exists(V1_PATH):
    df = pd.read_csv(V1_PATH, encoding='utf-8-sig')
    df['open_date'] = pd.to_datetime(df['open_date'], errors='coerce')
    print(f'✅ 기존 v1 CSV 로드 완료: {V1_PATH}')
    print(f'   shape: {df.shape[0]:,}편 × {df.shape[1]}컬럼')
elif VERSION != 'v1':
    raise FileNotFoundError(f'❌ {V1_PATH} 파일이 없습니다. VERSION="v1"로 먼저 Run All 하세요.')

✅ 기존 v1 CSV 로드 완료: data/processed/feature_table_v1.csv
   shape: 3,869편 × 27컬럼


In [7]:
# 8. 네이버 트렌드
trend_rows = []
try:
    trend_rows = db.fetch_all("""
        SELECT movie_id, trend_date, search_index
        FROM naver_search_trend
        ORDER BY movie_id, trend_date
    """)
    print(f'  ✅ trend_rows    : {len(trend_rows):,}건')
except Exception:
    print(f'  ⚠️  trend_rows    : 테이블 없음 → 0으로 채움')

print(f'\n✅ DB 로드 완료')

  ✅ trend_rows    : 80,371건

✅ DB 로드 완료


In [ ]:
# ── [셀 10] [v2] 네이버 검색 트렌드 피처 생성 및 v2 저장 ──

print('🔄 네이버 검색 트렌드 피처 계산 시작...')

# 1. 트렌드 데이터프레임 변환 및 전처리
df_trend = pd.DataFrame(trend_rows)
df_trend['trend_date'] = pd.to_datetime(df_trend['trend_date'])
df_trend['search_index'] = pd.to_numeric(df_trend['search_index'], errors='coerce').fillna(0)

# 빠른 룩업을 위한 맵 빌드
open_date_map = dict(zip(df['movie_id'], pd.to_datetime(df['open_date'])))

# 2. 개별 영화의 개봉 전 검색 트렌드 피처 연산 함수 정의
def calc_trend_features(movie_id, df_trend, open_date_map):
    open_dt = open_date_map.get(movie_id)
    if pd.isna(open_dt):
        return {
            'trend_pre7_avg': 0.0,
            'trend_pre7_max': 0.0,
            'trend_growth_rate': 0.0,
            'relative_search_share': 0.0
        }
    
    # 개봉 전(D-1일 이전) 데이터만 필터링 (누수 원천 차단)
    movie_data = df_trend[df_trend['movie_id'] == movie_id]
    pre_data = movie_data[movie_data['trend_date'] < open_dt]
    
    if pre_data.empty:
        return {
            'trend_pre7_avg': 0.0,
            'trend_pre7_max': 0.0,
            'trend_growth_rate': 0.0,
            'relative_search_share': 0.0
        }
        
    # D-7 ~ D-1 검색량 계산
    pre7 = pre_data[pre_data['trend_date'] >= open_dt - pd.Timedelta(days=7)]
    pre7_avg = float(pre7['search_index'].mean()) if not pre7.empty else 0.0
    pre7_max = float(pre7['search_index'].max()) if not pre7.empty else 0.0
    
    # D-30 ~ D-1 검색량 계산 (growth 및 share 계산용)
    pre30 = pre_data[pre_data['trend_date'] >= open_dt - pd.Timedelta(days=30)]
    pre30_avg = float(pre30['search_index'].mean()) if not pre30.empty else 0.0
    
    # 관심도 성장률 (D-7~D-1 평균 / D-30~D-1 평균)
    trend_growth_rate = (pre7_avg / pre30_avg) if pre30_avg > 0 else 0.0
    
    # 경쟁 영화 선정: 현재 영화 개봉일 기준 D-30 ~ D-1 이내에 개봉한 다른 영화들
    start_date = open_dt - pd.Timedelta(days=30)
    end_date = open_dt - pd.Timedelta(days=1)
    competing_movies = [
        mid for mid, dt in open_date_map.items()
        if mid != movie_id and not pd.isna(dt) and start_date <= dt <= end_date
    ]
    
    # 경쟁 영화들의 D-30 평균 검색지수 수집
    competing_pre30_avgs = []
    for mid in competing_movies:
        mid_open = open_date_map[mid]
        mid_trend = df_trend[(df_trend['movie_id'] == mid) & (df_trend['trend_date'] < mid_open)]
        if not mid_trend.empty:
            mid_pre30 = mid_trend[mid_trend['trend_date'] >= mid_open - pd.Timedelta(days=30)]
            if not mid_pre30.empty:
                competing_pre30_avgs.append(float(mid_pre30['search_index'].mean()))
    
    # 경쟁작 중 상위 Top 5 검색량의 합 구하기
    top5_avgs = sorted(competing_pre30_avgs, reverse=True)[:5]
    top5_sum = sum(top5_avgs)
    
    # 상대적 검색 점유율 계산
    relative_search_share = (pre30_avg / top5_sum) if top5_sum > 0 else 0.0
    
    return {
        'trend_pre7_avg': pre7_avg,
        'trend_pre7_max': pre7_max,
        'trend_growth_rate': trend_growth_rate,
        'relative_search_share': relative_search_share
    }

# 3. 전체 영화 대상 병합 연산 수행
trend_stats = []
total_movies = len(df)
for idx, mid in enumerate(df['movie_id']):
    stats = calc_trend_features(mid, df_trend, open_date_map)
    trend_stats.append({'movie_id': mid, **stats})
    
    if (idx + 1) % 500 == 0 or (idx + 1) == total_movies:
        print(f'   진행상황: {idx + 1}/{total_movies} 완료...')
        
df_trend_feats = pd.DataFrame(trend_stats)
df = df.merge(df_trend_feats, on='movie_id', how='left')

# 결측치 채우기 및 상태 모니터링 출력
TREND_COLS = ['trend_pre7_avg', 'trend_pre7_max', 'trend_growth_rate', 'relative_search_share']
df[TREND_COLS] = df[TREND_COLS].fillna(0)

print('✅ 네이버 트렌드 피처 생성 완료')
print(f'   trend_pre7_avg 평균: {df["trend_pre7_avg"].mean():.2f}')
print(f'   relative_search_share 평균: {df["relative_search_share"].mean():.4f}')

In [ ]:
print('🔄 v2 전처리 파이프라인 실행 시작...')

# [2] 장르 흥행력 기반 타겟 인코딩 (누수 없는 개봉 전 과거 기준)
print('   장르 흥행력 기반 과거 평균 관객수(타겟 인코딩) 계산 중...')
genre_past_means = []
for idx, row in df.iterrows():
    past_same_genre = df[
        (df['genre'] == row['genre']) & 
        (df['open_date'] < row['open_date'])
    ]
    if not past_same_genre.empty:
        genre_past_means.append(np.log1p(past_same_genre['total_audience'].mean()))
    else:
        # 과거 기록이 없는 경우 전체 Non-zero Median의 로그값 적용
        global_median = df.loc[df['total_audience'] > 0, 'total_audience'].median()
        genre_past_means.append(np.log1p(global_median))
        
df['genre_avg_audi'] = genre_past_means
print('   장르 타겟 인코딩')

# [3] 신인/영세 지시자(is_new_*) 생성 및 수치 피처 Non-zero 중위값 대체
impute_cols = ['director_avg_audi', 'lead_actor_avg_audi',
                'producer_avg_audi', 'distributor_avg_audi']
for col in impute_cols:
    non_zero_median = df.loc[df[col] > 0, col].median()
    if pd.isna(non_zero_median):
        non_zero_median = 10000.0
    prefix = col.split('_')[0]
    df[f'is_new_{prefix}'] = (df[col] == 0).astype(int)
    df.loc[df[col] == 0, col] = non_zero_median
print('   신인/영세 지시자(is_new_director 등 4종) 생성 및 중위값 보정 완료')

# [4] 수치 피처 로그 변환 (avg_audi 4종 + market_avg_audi_7d)
log_cols = ['director_avg_audi', 'lead_actor_avg_audi',
            'distributor_avg_audi', 'producer_avg_audi',
            'market_avg_audi_7d']
for col in log_cols:
    if col in df.columns:
        df[col] = np.log1p(df[col])
print('   주요 수치 피처 로그 변환(log1p) 완료')


In [ ]:
if VERSION == 'v2':
    print('\n💾 v2 설정 조건 만족 -> 파일 쓰기 및 저장 시작...')

    # [0] 학습 범위 필터: 2016년 이후 개봉 영화만 학습 대상으로 한정
    df['open_date'] = pd.to_datetime(df['open_date'], errors='coerce')
    df['open_year'] = df['open_date'].dt.year
    df = df[df['open_year'] >= 2016].reset_index(drop=True)
    print(f'   학습 범위 필터 적용 (2016년 이후): {len(df):,}편')
    
    _save_cols = ID_COLS + V1_FEATURE_COLS + TARGET_COLS + TREND_COLS
    _save_cols = [c for c in _save_cols if c in df.columns]
    
    df_out = df[_save_cols].copy()
    df_out['total_audience'] = df_out['total_audience'].astype(int)
    df_out['hit_class']      = df_out['hit_class'].astype(int)

    # [5] 범주형 원-핫 인코딩 (rating_encoded 원-핫, 장르는 수치형으로 인코딩 완료되어 자동 제외됨)
    # df_out = pd.get_dummies(df_out, columns=['rating_encoded'], drop_first=True, dtype=int)
    # print('   관람등급(rating_encoded) 원-핫 인코딩 완료')

    # 불필요 컬럼 삭제
    # drop_cols = ['open_month', 'cast_max_star_power',
    #             'director_movie_count', 'lead_actor_movie_count', 'genre',
    #             'distributor_movie_count', 'producer_movie_count']
    # df_out = df_out.drop(columns=[c for c in drop_cols if c in df_out.columns])
    # print(f'   불필요 피처 삭제 완료: {drop_cols}')
    
    df_out.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    print(f'✅ v2 저장 완료: {OUTPUT_PATH}')
    print(f'   shape: {df_out.shape[0]:,}편 × {df_out.shape[1]}컬럼')
    print(f'   컬럼 목록: {list(df_out.columns)}')
else:
    print(f'\nℹ️  VERSION={VERSION} → 파일 물리 저장을 건너뛰고 세션 메모리에서만 유지합니다.')